# Speaker verification with ESPnet-SPK

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/spk_demo.ipynb) [![Checked weekly](https://github.com/espnet/notebook/actions/workflows/run_notebooks.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/run_notebooks.yml)

Turn a recording into one vector that describes the voice rather than the
words. Two recordings of the same person land close together; two people
land far apart.

CPU, and the model is 25 MB.


## Install


In [ ]:
%pip install -q "espnet[spk]==202610.post1" espnet_model_zoo librosa


## Two recordings

Two different sentences, read by two different people.


In [ ]:
import librosa
from IPython.display import Audio, display

!wget -q -O a.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
!wget -q -O b.wav https://github.com/espnet/espnet/raw/master/test_utils/st_test.wav

for name in ("a.wav", "b.wav"):
    audio, rate = librosa.load(name, sr=16000)
    print(name)
    display(Audio(audio, rate=rate))


## An embedding each

[`voxcelebs12_rawnet3`](https://huggingface.co/espnet/voxcelebs12_rawnet3)
reads the waveform itself rather than a spectrogram, and reaches 0.739%
equal error rate on VoxCeleb1-O.


In [ ]:
import numpy as np
from espnet2.bin.spk_inference import Speech2Embedding

spk = Speech2Embedding.from_pretrained("espnet/voxcelebs12_rawnet3", device="cpu")

def embed(path):
    audio, _ = librosa.load(path, sr=16000)
    return spk(audio)[0].detach().cpu().numpy()

first, second = embed("a.wav"), embed("b.wav")
print("embedding size:", first.shape)


## Same speaker or not

The score is the cosine between the two vectors. Where to put the line is a
choice about which mistakes you would rather make, not something the model
records — so it is a number you pick, and 0.36 is only a place to start.


In [ ]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

# the same recording, cut in two: the same voice either way
audio, _ = librosa.load("a.wav", sr=16000)
half = len(audio) // 2
same = cosine(spk(audio[:half])[0].detach().numpy(),
              spk(audio[half:])[0].detach().numpy())
different = cosine(first, second)

print(f"same speaker      {same:.3f}")
print(f"different speakers {different:.3f}")


## Where next

- **In the browser**: the [speaker-verification Space](https://huggingface.co/spaces/espnet/speaker-verification) does this with
  two files you drop in
- **Other checkpoints**: every speaker model in the
  [espnet organization](https://huggingface.co/espnet?search=voxceleb) loads
  the same way
- **Diarization** — who spoke when, built on these embeddings:
  [`../Courses/CMUSpeechTechnology26S/speaker_verification.ipynb`](../Courses/CMUSpeechTechnology26S/speaker_verification.ipynb)
